# Customer Intelligence & Predictive Retention Platform
## Notebook 01 — Exploratory Data Analysis (EDA)

### Project Objective
This notebook explores customer churn patterns in the IBM Telco Customer Churn dataset. The goal is to understand customer characteristics, identify churn-related patterns, and derive actionable business insights.

### Dataset
- **Source:** IBM Telco Customer Churn dataset
- **Records:** 7,043 customers
- **Features:** 21 columns
- **Target variable:** `Churn`

### Analysis Performed
1. Dataset structure and data quality
2. Customer churn distribution
3. Contract type and churn rate
4. Customer tenure analysis
5. Monthly charges analysis
6. Business insights and retention recommendations

### Tools and Libraries
Python, Pandas, NumPy, Matplotlib, Seaborn, and Jupyter Notebook.

### Important Note
The relationships identified in this notebook are observational. They describe patterns in this dataset and do not establish that any single feature causes customer churn.


In [ ]:
# Customer Intelligence & Predictive Retention Platform
# Notebook 01: Exploratory Data Analysis

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

In [ ]:
# Load the original Telco Customer Churn dataset

DATA_PATH = Path("../data/raw/Telco-Customer-Churn.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

display(df.head())

In [ ]:
# Step 4: Dataset Overview and Data Quality

print("DATASET INFORMATION")
print("=" * 50)

print("\nDataset shape:")
print(df.shape)

print("\nColumn names and data types:")
df.info()

print("\nMissing values in each column:")
display(
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing Values")
)

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe(include="all").T)

In [ ]:
# Step 5: Customer Churn Distribution

if "Churn" not in df.columns:
    raise ValueError("The dataset does not contain a 'Churn' column.")

churn_counts = df["Churn"].value_counts()
churn_percentages = df["Churn"].value_counts(normalize=True) * 100

print("CUSTOMER CHURN SUMMARY")
display(pd.DataFrame({
    "Customer Count": churn_counts,
    "Percentage (%)": churn_percentages.round(2)
}))

plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="Churn")
ax.set_title("Customer Churn Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Number of Customers")

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Contract Type vs Customer Churn

contract_col = "Contract"

if contract_col not in df.columns:
    raise ValueError("Contract column not found in the dataset.")

contract_churn = pd.crosstab(
    df[contract_col],
    df["Churn"],
    normalize="index"
) * 100

print("CHURN RATE BY CONTRACT TYPE (%)")
display(contract_churn.round(2))

# Visualize churn rates by contract
churn_rate = (
    df.assign(ChurnFlag=(df["Churn"] == "Yes").astype(int))
      .groupby(contract_col)["ChurnFlag"]
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=churn_rate.index, y=churn_rate.values)

ax.set_title("Customer Churn Rate by Contract Type")
ax.set_xlabel("Contract Type")
ax.set_ylabel("Churn Rate (%)")
ax.bar_label(ax.containers[0], fmt="%.1f%%")

plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Customer Tenure vs Churn

if "tenure" not in df.columns:
    raise ValueError("tenure column not found.")

tenure_summary = (
    df.groupby("Churn")["tenure"]
      .agg(["count", "mean", "median"])
      .round(2)
)

print("TENURE SUMMARY BY CHURN STATUS")
display(tenure_summary)

plt.figure(figsize=(9, 5))
sns.histplot(
    data=df,
    x="tenure",
    hue="Churn",
    bins=30,
    multiple="stack"
)

plt.title("Customer Tenure Distribution by Churn")
plt.xlabel("Tenure (months)")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

In [ ]:
# Step 8: Monthly Charges vs Customer Churn

if "MonthlyCharges" not in df.columns:
    raise ValueError("MonthlyCharges column not found.")

charges_summary = (
    df.groupby("Churn")["MonthlyCharges"]
      .agg(["count", "mean", "median"])
      .round(2)
)

print("MONTHLY CHARGES BY CHURN STATUS")
display(charges_summary)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")

plt.title("Monthly Charges by Customer Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges ($)")
plt.tight_layout()
plt.show()

In [ ]:
# Step 9: Automated Business Insights Summary

churn_rate = (df["Churn"] == "Yes").mean() * 100

avg_tenure = df.groupby("Churn")["tenure"].mean()
avg_charges = df.groupby("Churn")["MonthlyCharges"].mean()

contract_rates = (
    df.assign(ChurnFlag=(df["Churn"] == "Yes").astype(int))
      .groupby("Contract")["ChurnFlag"]
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

print("BUSINESS INSIGHTS")
print("=" * 50)
print(f"Total customers: {len(df):,}")
print(f"Overall churn rate: {churn_rate:.2f}%")
print(f"Average tenure - Churned: {avg_tenure['Yes']:.2f} months")
print(f"Average tenure - Retained: {avg_tenure['No']:.2f} months")
print(f"Average monthly charges - Churned: ${avg_charges['Yes']:.2f}")
print(f"Average monthly charges - Retained: ${avg_charges['No']:.2f}")

print("\nChurn rate by contract:")
display(contract_rates.round(2).rename("Churn Rate (%)").to_frame())

print("\nRecommended business actions:")
print("1. Review retention offers for month-to-month customers.")
print("2. Investigate early-tenure customer experience.")
print("3. Review pricing and service concerns among high-charge customers.")
print("4. Validate these findings before launching retention campaigns.")